<a href="https://colab.research.google.com/github/alfredqbit/grcshjepa/blob/main/GR_CS_HJEPA_Chapter4_Phase1_Pilot_Hardening_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GR-CS-HJEPA Chapter 4 — Phase 1 Pilot Hardening Colab Notebook

This notebook is the next step **after smoke experiments pass**. It does not produce confirmatory dissertation findings. It creates and runs a disciplined Phase 1 pilot workflow whose purpose is to estimate runtime, variance, failure modes, metric sanity, and readiness for freezing a confirmatory protocol.

Expected inputs:

1. The `grcshjepa` GitHub repository created from the smoke-tested scaffold.
2. The Phase 0 smoke notebook already passed unit tests and smoke experiments.
3. Colab Pro/Pro+ or another Python runtime with enough memory for modest pilot runs.

Main outputs:

- `configs/*_pilot.yaml`
- `src/grcshjepa/pilot/runner.py`
- `src/grcshjepa/pilot/analysis.py`
- `scripts/run_phase1_pilot.py`
- `scripts/analyze_phase1_pilot.py`
- `runs/phase1_pilot/`
- `analysis/phase1_pilot/phase1_pilot_readiness_report.md`
- a timestamped archive copied to Google Drive when Drive is mounted

**Important:** pilot results are not dissertation results. They are used to decide what must be fixed before confirmatory runs.

In [1]:
# ============================================================
# 1. Runtime setup: Colab/Drive, project directory, and options
# ============================================================
from __future__ import annotations

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ---- USER SETTINGS -------------------------------------------------
# Use your repository. The default below mirrors the repo used in the prior smoke notebook.
REPO_URL = "https://github.com/alfredqbit/grcshjepa.git"  # change if needed
PROJECT_DIR = Path("/content/grcshjepa") if IN_COLAB else Path.cwd() / "grcshjepa"
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/grcshjepa_artifacts") if IN_COLAB else None

# This notebook writes Phase 1 pilot files. It should not rewrite the original smoke scaffold.
WRITE_PHASE1_FILES = True
OVERWRITE_PHASE1_FILES = True

# Run controls. For a first pass, keep max seeds at 4 and all studies enabled.
RUN_UNIT_TESTS = True
RUN_PILOT_STUDIES = True
RUN_ANALYSIS = True
RUN_ARCHIVE_TO_DRIVE = True
MAX_SEEDS = None  # set to 1 for a quick debugging pass
STUDIES = "study1,study2,study3"  # comma-separated subset allowed

print({
    "IN_COLAB": IN_COLAB,
    "PROJECT_DIR": str(PROJECT_DIR),
    "DRIVE_ARTIFACT_DIR": str(DRIVE_ARTIFACT_DIR) if DRIVE_ARTIFACT_DIR else None,
    "UTC": datetime.now(timezone.utc).isoformat(),
})

Mounted at /content/drive
{'IN_COLAB': True, 'PROJECT_DIR': '/content/grcshjepa', 'DRIVE_ARTIFACT_DIR': '/content/drive/MyDrive/grcshjepa_artifacts', 'UTC': '2026-07-26T23:15:50.815979+00:00'}


In [2]:
# ============================================================
# 2. Clone or update the GitHub repository
# ============================================================
def run(cmd: str, cwd: Path | None = None, check: bool = True):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with status {proc.returncode}: {cmd}")
    return proc

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    run("git pull", cwd=PROJECT_DIR, check=False)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        print(f"Project directory exists and is not empty: {PROJECT_DIR}")
        print("Using it as-is. If this is wrong, set PROJECT_DIR differently or remove the folder.")
    else:
        PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
        run(f"git clone {REPO_URL} {PROJECT_DIR}", check=False)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())
print("Repository files:")
for p in sorted(Path.cwd().glob("*"))[:30]:
    print(" ", p)

# Guardrail: this notebook assumes the Phase 0 scaffold exists.
required = [Path("pyproject.toml"), Path("src/grcshjepa"), Path("tests"), Path("configs")]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required scaffold paths: " + ", ".join(missing) +
        "\nRun the Phase 0 Colab driver first or clone the correct repository."
    )

$ git clone https://github.com/alfredqbit/grcshjepa.git /content/grcshjepa
Cloning into '/content/grcshjepa'...

Working directory: /content/grcshjepa
Repository files:
  /content/grcshjepa/.git
  /content/grcshjepa/GR_CS_HJEPA_Chapter4_Colab_Driver.ipynb
  /content/grcshjepa/LICENSE
  /content/grcshjepa/README.md
  /content/grcshjepa/configs
  /content/grcshjepa/pyproject.toml
  /content/grcshjepa/scripts
  /content/grcshjepa/src
  /content/grcshjepa/tests


In [3]:
# ============================================================
# 3. Write Phase 1 pilot configs, modules, scripts, and tests
# ============================================================
from pathlib import Path

PHASE1_FILES = {"configs/phase1_pilot.yaml": "# Meta-configuration for Phase 1 pilot hardening.\n# These pilot runs are for variance/runtime/failure profiling only, not confirmatory results.\nphase: phase1_pilot\nseed_list: [0, 1, 2, 3]\nstudies: [study1, study2, study3]\nstudy1_config: configs/study1_pilot.yaml\nstudy2_config: configs/study2_pilot.yaml\nstudy3_config: configs/study3_pilot.yaml\noutput_dir: runs/phase1_pilot\narchive_name: phase1_pilot.tar.gz\n", "configs/study1_pilot.yaml": "project: GR-CS-HJEPA-Chapter4\nsmoke_mode: false\nseed: 0\ndevice: auto\noutput_dir: runs/phase1_pilot\n\n# Procedural data: still modest, but no longer toy smoke size.\nmaze_size_train: 10\nmaze_size_ood: 14\nmaze_obstacle_prob: 0.20\nsorting_length_train: 10\nsorting_length_ood: 14\ntrain_samples: 384\nval_samples: 128\ntest_samples: 128\n\n# H-JEPA pilot settings.\nlatent_dim: 48\nprojection_dim: 16\nhorizon_set: [1, 2, 3]\nbatch_size: 32\nepochs: 3\nlr: 0.0015\nema_tau: 0.99\nlambda_ac: 0.03\nlambda_suff: 0.0\nlambda_alias: 0.0\nlambda_unc: 0.0\n\n# Relaxed continuous-spiking predictor.\nn_neurons: 64\nn_filters: 2\ninternal_steps: 6\ndt: 0.08\nnoise_std: 0.01\nsurrogate_eps: 0.7\n\n# Downstream heads retained for config compatibility.\ndownstream_label_fraction: 0.15\nhead_epochs: 4\nhead_lr: 0.0015\n\n# Routing retained for config compatibility.\nrouting_nodes: 48\nrouting_segments: 128\nrouting_terminals: 12\ndamage_levels: [0.05, 0.10, 0.15]\n", "configs/study2_pilot.yaml": "project: GR-CS-HJEPA-Chapter4\nsmoke_mode: false\nseed: 0\ndevice: auto\noutput_dir: runs/phase1_pilot\n\nmaze_size_train: 10\nmaze_size_ood: 14\nmaze_obstacle_prob: 0.20\nsorting_length_train: 10\nsorting_length_ood: 14\ntrain_samples: 384\nval_samples: 128\ntest_samples: 128\n\nlatent_dim: 48\nprojection_dim: 16\nhorizon_set: [1, 2, 3]\nbatch_size: 32\nepochs: 3\nlr: 0.0015\nema_tau: 0.99\nlambda_ac: 0.03\nlambda_suff: 0.0\nlambda_alias: 0.0\nlambda_unc: 0.0\n\nn_neurons: 64\nn_filters: 2\ninternal_steps: 6\ndt: 0.08\nnoise_std: 0.01\nsurrogate_eps: 0.7\n\n# Low-shot downstream-head pilot.\ndownstream_label_fraction: 0.10\nhead_epochs: 8\nhead_lr: 0.0015\n\nrouting_nodes: 48\nrouting_segments: 128\nrouting_terminals: 12\ndamage_levels: [0.05, 0.10, 0.15]\n", "configs/study3_pilot.yaml": "project: GR-CS-HJEPA-Chapter4\nsmoke_mode: false\nseed: 0\ndevice: auto\noutput_dir: runs/phase1_pilot\n\n# Data/model fields retained for ExperimentConfig compatibility.\nmaze_size_train: 10\nmaze_size_ood: 14\nmaze_obstacle_prob: 0.20\nsorting_length_train: 10\nsorting_length_ood: 14\ntrain_samples: 128\nval_samples: 64\ntest_samples: 64\nlatent_dim: 48\nprojection_dim: 16\nhorizon_set: [1, 2, 3]\nbatch_size: 32\nepochs: 1\nlr: 0.0015\nema_tau: 0.99\nlambda_ac: 0.03\nlambda_suff: 0.0\nlambda_alias: 0.0\nlambda_unc: 0.0\nn_neurons: 64\nn_filters: 2\ninternal_steps: 6\ndt: 0.08\nnoise_std: 0.01\nsurrogate_eps: 0.7\ndownstream_label_fraction: 0.10\nhead_epochs: 4\nhead_lr: 0.0015\n\n# Routing pilot: larger than smoke, still fast enough for Colab.\nrouting_nodes: 96\nrouting_segments: 256\nrouting_terminals: 16\ndamage_levels: [0.05, 0.10, 0.15]\n", "src/grcshjepa/pilot/__init__.py": "\"\"\"Phase 1 pilot hardening utilities for GR-CS-HJEPA.\"\"\"\n", "src/grcshjepa/pilot/runner.py": "\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport time\nimport traceback\nfrom pathlib import Path\nfrom typing import Callable, Iterable\n\nimport pandas as pd\nimport torch\nimport yaml\nfrom torch.utils.data import DataLoader\n\nfrom grcshjepa.config import ExperimentConfig\nfrom grcshjepa.data.datasets import MazeTrajectoryDataset, SortingTrajectoryDataset\nfrom grcshjepa.models.encoders import MazeEncoder, SortingEncoder\nfrom grcshjepa.models.hjepa import HJEPA\nfrom grcshjepa.models.predictors import MLPPredictor, RelaxedContinuousSpikingPredictor\nfrom grcshjepa.models.heads import MazeActionHead, SortingHead\nfrom grcshjepa.routing.graph import make_toy_routing_graph\nfrom grcshjepa.routing.surface import routing_surface\nfrom grcshjepa.routing.damage import apply_damage\nfrom grcshjepa.training.pretrain import train_jepa_epoch, evaluate_jepa\nfrom grcshjepa.training.downstream import (\n    make_labeled_subset,\n    encode_dataset,\n    train_classification_head,\n    evaluate_classification_head,\n    train_regression_head,\n    evaluate_regression_head,\n)\nfrom grcshjepa.utils import (\n    archive_directory,\n    config_hash,\n    environment_report,\n    resolve_device,\n    save_json,\n    set_seed,\n    write_manifest,\n)\n\n\ndef load_yaml(path: str | Path) -> dict:\n    path = Path(path)\n    data = yaml.safe_load(path.read_text()) if path.exists() else {}\n    return {} if data is None else dict(data)\n\n\ndef _fresh_encoder(task: str, cfg: ExperimentConfig):\n    if task == \"maze\":\n        return MazeEncoder(cfg.latent_dim)\n    if task == \"sorting\":\n        return SortingEncoder(cfg.sorting_length_train, cfg.latent_dim)\n    raise ValueError(f\"Unknown task: {task}\")\n\n\ndef _make_hjepa(cfg: ExperimentConfig, task: str, predictor_type: str) -> HJEPA:\n    max_horizon = max(cfg.horizon_set)\n    horizon_dim = 16\n    encoder = _fresh_encoder(task, cfg)\n    if predictor_type == \"spiking\":\n        predictor = RelaxedContinuousSpikingPredictor(\n            latent_dim=cfg.latent_dim,\n            horizon_dim=horizon_dim,\n            n_neurons=cfg.n_neurons,\n            n_filters=cfg.n_filters,\n            internal_steps=cfg.internal_steps,\n            dt=cfg.dt,\n            noise_std=cfg.noise_std,\n            surrogate_eps=cfg.surrogate_eps,\n        )\n    elif predictor_type == \"mlp\":\n        predictor = MLPPredictor(latent_dim=cfg.latent_dim, horizon_dim=horizon_dim)\n    else:\n        raise ValueError(f\"Unknown predictor_type: {predictor_type}\")\n    return HJEPA(\n        encoder=encoder,\n        latent_dim=cfg.latent_dim,\n        projection_dim=cfg.projection_dim,\n        max_horizon=max_horizon,\n        predictor=predictor,\n        horizon_dim=horizon_dim,\n    )\n\n\ndef _dataset(task: str, cfg: ExperimentConfig, split: str, seed_offset: int):\n    seed = cfg.seed + seed_offset\n    if task == \"maze\":\n        n = {\"train\": cfg.train_samples, \"val\": cfg.val_samples, \"test\": cfg.test_samples}[split]\n        return MazeTrajectoryDataset(n, cfg.maze_size_train, cfg.maze_obstacle_prob, seed, cfg.horizon_set)\n    if task == \"sorting\":\n        n = {\"train\": cfg.train_samples, \"val\": cfg.val_samples, \"test\": cfg.test_samples}[split]\n        return SortingTrajectoryDataset(n, cfg.sorting_length_train, seed, cfg.horizon_set)\n    raise ValueError(f\"Unknown task: {task}\")\n\n\ndef _manifest_payload(study: str, cfg: ExperimentConfig, rows: int, status: str, started: float, failure: str | None = None) -> dict:\n    return {\n        \"phase\": \"phase1_pilot\",\n        \"study\": study,\n        \"seed\": cfg.seed,\n        \"status\": status,\n        \"rows\": rows,\n        \"config_hash\": config_hash(cfg),\n        \"config\": cfg.to_dict(),\n        \"runtime_seconds\": round(time.time() - started, 3),\n        \"failure\": failure,\n        \"environment\": environment_report(),\n    }\n\n\ndef run_study1_pilot_seed(cfg: ExperimentConfig) -> pd.DataFrame:\n    \"\"\"Pilot Study 1: predictive pretraining diagnostics across task and predictor variants.\"\"\"\n    started = time.time()\n    set_seed(cfg.seed)\n    device = resolve_device(cfg.device)\n    outdir = Path(cfg.output_dir) / \"study1\" / f\"seed_{cfg.seed:03d}\"\n    outdir.mkdir(parents=True, exist_ok=True)\n    rows: list[dict] = []\n\n    for task in [\"maze\", \"sorting\"]:\n        train_ds = _dataset(task, cfg, \"train\", 11 if task == \"maze\" else 21)\n        val_ds = _dataset(task, cfg, \"val\", 111 if task == \"maze\" else 211)\n        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)\n        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)\n\n        for predictor_type in [\"mlp\", \"spiking\"]:\n            model = _make_hjepa(cfg, task, predictor_type).to(device)\n            opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)\n            metrics: dict[str, float] = {}\n            for epoch in range(cfg.epochs):\n                train_metrics = train_jepa_epoch(\n                    model, train_loader, opt, device,\n                    lambda_ac=cfg.lambda_ac,\n                    ema_tau=cfg.ema_tau,\n                )\n                val_metrics = evaluate_jepa(model, val_loader, device)\n                metrics = {\n                    **{f\"train_{k}\": v for k, v in train_metrics.items()},\n                    **{f\"val_{k}\": v for k, v in val_metrics.items()},\n                }\n            rows.append({\n                \"phase\": \"phase1_pilot\",\n                \"study\": \"study1\",\n                \"task\": task,\n                \"predictor\": predictor_type,\n                \"seed\": cfg.seed,\n                \"status\": \"complete\",\n                \"epochs\": cfg.epochs,\n                \"train_samples\": cfg.train_samples,\n                \"val_samples\": cfg.val_samples,\n                **metrics,\n            })\n\n    df = pd.DataFrame(rows)\n    df.to_csv(outdir / f\"study1_seed{cfg.seed:03d}.csv\", index=False)\n    write_manifest(outdir / f\"manifest_seed{cfg.seed:03d}.json\", _manifest_payload(\"study1\", cfg, len(df), \"complete\", started))\n    return df\n\n\ndef _pretrain_for_downstream(cfg: ExperimentConfig, task: str, device: torch.device) -> HJEPA:\n    train_ds = _dataset(task, cfg, \"train\", 31 if task == \"maze\" else 41)\n    loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)\n    model = _make_hjepa(cfg, task, \"spiking\").to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr)\n    for _ in range(cfg.epochs):\n        train_jepa_epoch(model, loader, opt, device, lambda_ac=cfg.lambda_ac, ema_tau=cfg.ema_tau)\n    return model\n\n\ndef run_study2_pilot_seed(cfg: ExperimentConfig) -> pd.DataFrame:\n    \"\"\"Pilot Study 2: frozen-backbone downstream heads under limited labels.\"\"\"\n    started = time.time()\n    set_seed(cfg.seed)\n    device = resolve_device(cfg.device)\n    outdir = Path(cfg.output_dir) / \"study2\" / f\"seed_{cfg.seed:03d}\"\n    outdir.mkdir(parents=True, exist_ok=True)\n    rows: list[dict] = []\n\n    # Maze action head.\n    maze_model = _pretrain_for_downstream(cfg, \"maze\", device)\n    train_maze = _dataset(\"maze\", cfg, \"train\", 51)\n    test_maze = _dataset(\"maze\", cfg, \"test\", 52)\n    labeled_maze = make_labeled_subset(train_maze, cfg.downstream_label_fraction, cfg.seed)\n    z_train, y_train = encode_dataset(maze_model.encoder, DataLoader(labeled_maze, batch_size=cfg.batch_size), device, \"action\")\n    z_test, y_test = encode_dataset(maze_model.encoder, DataLoader(test_maze, batch_size=cfg.batch_size), device, \"action\")\n    head = MazeActionHead(cfg.latent_dim)\n    train_m = train_classification_head(head, z_train, y_train, epochs=cfg.head_epochs, lr=cfg.head_lr)\n    test_m = evaluate_classification_head(head, z_test, y_test)\n    rows.append({\n        \"phase\": \"phase1_pilot\",\n        \"study\": \"study2\",\n        \"task\": \"maze_action\",\n        \"head_type\": \"classification\",\n        \"seed\": cfg.seed,\n        \"status\": \"complete\",\n        \"label_fraction\": cfg.downstream_label_fraction,\n        \"head_epochs\": cfg.head_epochs,\n        **train_m,\n        **test_m,\n    })\n\n    # Sorting regression head.\n    sort_model = _pretrain_for_downstream(cfg, \"sorting\", device)\n    train_sort = _dataset(\"sorting\", cfg, \"train\", 61)\n    test_sort = _dataset(\"sorting\", cfg, \"test\", 62)\n    labeled_sort = make_labeled_subset(train_sort, cfg.downstream_label_fraction, cfg.seed)\n    z_train, y_train = encode_dataset(sort_model.encoder, DataLoader(labeled_sort, batch_size=cfg.batch_size), device, \"y_sorted\")\n    z_test, y_test = encode_dataset(sort_model.encoder, DataLoader(test_sort, batch_size=cfg.batch_size), device, \"y_sorted\")\n    head = SortingHead(cfg.latent_dim, cfg.sorting_length_train)\n    train_m = train_regression_head(head, z_train, y_train, epochs=cfg.head_epochs, lr=cfg.head_lr)\n    test_m = evaluate_regression_head(head, z_test, y_test)\n    rows.append({\n        \"phase\": \"phase1_pilot\",\n        \"study\": \"study2\",\n        \"task\": \"sorting_head\",\n        \"head_type\": \"regression\",\n        \"seed\": cfg.seed,\n        \"status\": \"complete\",\n        \"label_fraction\": cfg.downstream_label_fraction,\n        \"head_epochs\": cfg.head_epochs,\n        **train_m,\n        **test_m,\n    })\n\n    df = pd.DataFrame(rows)\n    df.to_csv(outdir / f\"study2_seed{cfg.seed:03d}.csv\", index=False)\n    write_manifest(outdir / f\"manifest_seed{cfg.seed:03d}.json\", _manifest_payload(\"study2\", cfg, len(df), \"complete\", started))\n    return df\n\n\ndef run_study3_pilot_seed(cfg: ExperimentConfig) -> pd.DataFrame:\n    \"\"\"Pilot Study 3: routing-surface metrics and cable-damage interventions.\"\"\"\n    started = time.time()\n    set_seed(cfg.seed)\n    outdir = Path(cfg.output_dir) / \"study3\" / f\"seed_{cfg.seed:03d}\"\n    outdir.mkdir(parents=True, exist_ok=True)\n    rows: list[dict] = []\n\n    variants = [\"sparsity\", \"euclidean_length\", \"tube_only\", \"full_surface\"]\n    damage_types = [\"uniform\", \"spatial\", \"load_targeted\"]\n    for variant in variants:\n        graph = make_toy_routing_graph(cfg.routing_nodes, cfg.routing_segments, cfg.seed, variant)\n        base = routing_surface(graph)\n        rows.append({\n            \"phase\": \"phase1_pilot\",\n            \"study\": \"study3\",\n            \"variant\": variant,\n            \"damage_type\": \"none\",\n            \"damage_level\": 0.0,\n            \"seed\": cfg.seed,\n            \"status\": \"complete\",\n            **base,\n        })\n        base_norm = float(base.get(\"normalized_surface\", float(\"nan\")))\n        base_traffic = float(base.get(\"delivered_traffic\", float(\"nan\")))\n        for dtype in damage_types:\n            for level in cfg.damage_levels:\n                damaged = apply_damage(graph, dtype, level, seed=cfg.seed + int(level * 1000) + len(dtype))\n                metrics = routing_surface(damaged)\n                metrics[\"surface_degradation\"] = float(metrics[\"normalized_surface\"] - base_norm)\n                metrics[\"traffic_degradation\"] = float(base_traffic - metrics[\"delivered_traffic\"])\n                rows.append({\n                    \"phase\": \"phase1_pilot\",\n                    \"study\": \"study3\",\n                    \"variant\": variant,\n                    \"damage_type\": dtype,\n                    \"damage_level\": float(level),\n                    \"seed\": cfg.seed,\n                    \"status\": \"complete\",\n                    **metrics,\n                })\n\n    df = pd.DataFrame(rows)\n    df.to_csv(outdir / f\"study3_seed{cfg.seed:03d}.csv\", index=False)\n    write_manifest(outdir / f\"manifest_seed{cfg.seed:03d}.json\", _manifest_payload(\"study3\", cfg, len(df), \"complete\", started))\n    return df\n\n\ndef _failure_frame(study: str, seed: int, error: BaseException) -> pd.DataFrame:\n    return pd.DataFrame([{\n        \"phase\": \"phase1_pilot\",\n        \"study\": study,\n        \"seed\": seed,\n        \"status\": \"failure\",\n        \"failure_type\": type(error).__name__,\n        \"failure_message\": str(error),\n    }])\n\n\ndef _run_one_safely(study: str, cfg: ExperimentConfig, fn: Callable[[ExperimentConfig], pd.DataFrame]) -> pd.DataFrame:\n    started = time.time()\n    outdir = Path(cfg.output_dir) / study / f\"seed_{cfg.seed:03d}\"\n    outdir.mkdir(parents=True, exist_ok=True)\n    try:\n        return fn(cfg)\n    except BaseException as exc:  # keep pilot loop going and record the failure\n        tb = traceback.format_exc()\n        (outdir / f\"failure_seed{cfg.seed:03d}.txt\").write_text(tb)\n        write_manifest(outdir / f\"manifest_seed{cfg.seed:03d}.json\", _manifest_payload(study, cfg, 1, \"failure\", started, tb))\n        df = _failure_frame(study, cfg.seed, exc)\n        df.to_csv(outdir / f\"{study}_seed{cfg.seed:03d}_failure.csv\", index=False)\n        return df\n\n\ndef run_phase1_pilot(\n    meta_config_path: str | Path,\n    output_dir: str | Path | None = None,\n    studies_override: Iterable[str] | None = None,\n    max_seeds: int | None = None,\n    archive: bool = False,\n) -> pd.DataFrame:\n    meta = load_yaml(meta_config_path)\n    root = Path(output_dir or meta.get(\"output_dir\", \"runs/phase1_pilot\"))\n    root.mkdir(parents=True, exist_ok=True)\n    studies = list(studies_override or meta.get(\"studies\", [\"study1\", \"study2\", \"study3\"]))\n    seeds = [int(s) for s in meta.get(\"seed_list\", [0, 1, 2, 3])]\n    if max_seeds is not None:\n        seeds = seeds[: int(max_seeds)]\n\n    config_key = {\n        \"study1\": \"study1_config\",\n        \"study2\": \"study2_config\",\n        \"study3\": \"study3_config\",\n    }\n    runner = {\n        \"study1\": run_study1_pilot_seed,\n        \"study2\": run_study2_pilot_seed,\n        \"study3\": run_study3_pilot_seed,\n    }\n\n    frames: list[pd.DataFrame] = []\n    for seed in seeds:\n        for study in studies:\n            if study not in runner:\n                raise ValueError(f\"Unknown study: {study}\")\n            cfg_path = meta[config_key[study]]\n            cfg = ExperimentConfig.from_yaml(cfg_path).with_updates(seed=seed, output_dir=str(root))\n            print(f\"[phase1] running {study} seed={seed} config_hash={config_hash(cfg)}\")\n            df = _run_one_safely(study, cfg, runner[study])\n            frames.append(df)\n\n    combined = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()\n    combined_path = root / \"phase1_pilot_combined_results.csv\"\n    combined.to_csv(combined_path, index=False)\n    write_manifest(root / \"phase1_pilot_manifest.json\", {\n        \"phase\": \"phase1_pilot\",\n        \"status\": \"complete\",\n        \"meta_config\": meta,\n        \"seed_list\": seeds,\n        \"studies\": studies,\n        \"rows\": len(combined),\n        \"combined_results\": str(combined_path),\n        \"environment\": environment_report(),\n    })\n    if archive:\n        archive_name = meta.get(\"archive_name\", \"phase1_pilot.tar.gz\")\n        archive_path = archive_directory(root, root.parent / archive_name)\n        print(f\"[phase1] archived to {archive_path}\")\n    return combined\n\n\ndef _parse_studies(value: str | None):\n    if value is None or value.strip() == \"\":\n        return None\n    return [part.strip() for part in value.split(\",\") if part.strip()]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"Run Phase 1 pilot hardening experiments.\")\n    parser.add_argument(\"--meta-config\", default=\"configs/phase1_pilot.yaml\")\n    parser.add_argument(\"--output-dir\", default=None)\n    parser.add_argument(\"--studies\", default=None, help=\"Comma-separated subset, e.g. study1,study3\")\n    parser.add_argument(\"--max-seeds\", type=int, default=None, help=\"Use only the first N seeds for quick checks\")\n    parser.add_argument(\"--archive\", action=\"store_true\")\n    args = parser.parse_args()\n    df = run_phase1_pilot(\n        args.meta_config,\n        output_dir=args.output_dir,\n        studies_override=_parse_studies(args.studies),\n        max_seeds=args.max_seeds,\n        archive=args.archive,\n    )\n    print(df.head(40).to_string(index=False))\n    print(f\"Rows written: {len(df)}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "src/grcshjepa/pilot/analysis.py": "\nfrom __future__ import annotations\n\nimport argparse\nimport math\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nfrom grcshjepa.stats.decision import bootstrap_ci\nfrom grcshjepa.utils import save_json\n\n\nID_COLUMNS = {\n    \"phase\", \"study\", \"task\", \"predictor\", \"variant\", \"damage_type\", \"damage_level\",\n    \"head_type\", \"seed\", \"status\", \"failure_type\", \"failure_message\",\n}\n\n\ndef load_pilot_results(results_dir: str | Path) -> pd.DataFrame:\n    results_dir = Path(results_dir)\n    preferred = results_dir / \"phase1_pilot_combined_results.csv\"\n    if preferred.exists():\n        return pd.read_csv(preferred)\n    csvs = sorted(results_dir.glob(\"**/*.csv\"))\n    frames = []\n    for path in csvs:\n        if \"summary\" in path.name.lower() or \"readiness\" in path.name.lower():\n            continue\n        try:\n            df = pd.read_csv(path)\n            df[\"source_csv\"] = str(path)\n            frames.append(df)\n        except Exception:\n            pass\n    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()\n\n\ndef _group_columns(df: pd.DataFrame) -> list[str]:\n    candidates = [\"study\", \"task\", \"predictor\", \"variant\", \"damage_type\", \"damage_level\", \"head_type\"]\n    return [c for c in candidates if c in df.columns and df[c].notna().any()]\n\n\ndef summarize_numeric(df: pd.DataFrame) -> pd.DataFrame:\n    if df.empty:\n        return pd.DataFrame()\n    group_cols = _group_columns(df)\n    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in {\"seed\"}]\n    if not numeric_cols:\n        return pd.DataFrame()\n    rows = []\n    grouped = df.groupby(group_cols, dropna=False) if group_cols else [((), df)]\n    for key, g in grouped:\n        base = dict(zip(group_cols, key if isinstance(key, tuple) else (key,))) if group_cols else {}\n        for col in numeric_cols:\n            vals = g[col].dropna().astype(float).to_numpy()\n            if vals.size == 0:\n                continue\n            lo, hi = bootstrap_ci(vals, n_boot=1000, seed=17) if vals.size >= 2 else (float(\"nan\"), float(\"nan\"))\n            rows.append({\n                **base,\n                \"metric\": col,\n                \"n_rows\": int(vals.size),\n                \"n_seeds\": int(g[\"seed\"].nunique()) if \"seed\" in g else int(vals.size),\n                \"mean\": float(np.mean(vals)),\n                \"sd\": float(np.std(vals, ddof=1)) if vals.size >= 2 else 0.0,\n                \"median\": float(np.median(vals)),\n                \"q25\": float(np.quantile(vals, 0.25)),\n                \"q75\": float(np.quantile(vals, 0.75)),\n                \"boot_ci_low\": lo,\n                \"boot_ci_high\": hi,\n            })\n    return pd.DataFrame(rows)\n\n\ndef _suggest_n(sd: float, target_half_width: float, max_n: int = 20) -> int | None:\n    if not np.isfinite(sd) or sd <= 0 or target_half_width <= 0:\n        return None\n    n = math.ceil((1.96 * sd / target_half_width) ** 2)\n    return int(min(max(n, 2), max_n))\n\n\ndef readiness_report(df: pd.DataFrame, summary: pd.DataFrame) -> dict:\n    report: dict = {\n        \"phase\": \"phase1_pilot\",\n        \"interpretation\": \"Pilot hardening only. These values estimate runtime, variance, failure modes, and metric sanity; they are not confirmatory dissertation results.\",\n        \"rows\": int(len(df)),\n        \"studies_present\": sorted([str(x) for x in df.get(\"study\", pd.Series(dtype=str)).dropna().unique()]),\n        \"seeds_present\": sorted([int(x) for x in df.get(\"seed\", pd.Series(dtype=float)).dropna().unique()]) if \"seed\" in df else [],\n        \"status_counts\": df.get(\"status\", pd.Series(dtype=str)).fillna(\"unknown\").value_counts().to_dict() if not df.empty else {},\n        \"failure_rate\": None,\n        \"suggested_confirmatory_n\": [],\n        \"gate_checks\": [],\n        \"next_actions\": [\n            \"Inspect failure manifests and rerun only administrative failures under the same seed.\",\n            \"Replace remaining toy/smoke stand-ins with production architecture modules where needed.\",\n            \"Freeze model variants, primary endpoints, seed list, hyperparameter budget, and analysis scripts before confirmatory runs.\",\n            \"Use pilot variance to select 12 to 20 independent confirmatory seeds per primary variant.\",\n        ],\n    }\n    if not df.empty and \"status\" in df:\n        n = len(df)\n        failures = int((df[\"status\"].fillna(\"\") == \"failure\").sum())\n        report[\"failure_rate\"] = failures / max(1, n)\n\n    if not summary.empty:\n        # Heuristic target half-widths for pilot planning only.\n        target_by_metric = {\n            \"val_pred_loss\": 0.02,\n            \"test_acc\": 0.03,\n            \"test_mse\": 0.02,\n            \"exactish_rate\": 0.03,\n            \"normalized_surface\": 0.05,\n            \"traffic_degradation\": 1.0,\n            \"surface_degradation\": 0.05,\n        }\n        for _, row in summary.iterrows():\n            metric = row.get(\"metric\")\n            if metric in target_by_metric and row.get(\"n_seeds\", 0) >= 2:\n                n_suggest = _suggest_n(float(row.get(\"sd\", float(\"nan\"))), target_by_metric[metric])\n                if n_suggest is not None:\n                    item = {k: row[k] for k in [\"study\", \"task\", \"predictor\", \"variant\", \"damage_type\", \"damage_level\", \"head_type\"] if k in row and pd.notna(row[k])}\n                    item.update({\"metric\": metric, \"pilot_sd\": float(row[\"sd\"]), \"target_half_width\": target_by_metric[metric], \"suggested_n_cap20\": n_suggest})\n                    report[\"suggested_confirmatory_n\"].append(item)\n\n    # Coarse gate checks. Human review is still required.\n    if report[\"failure_rate\"] is not None:\n        report[\"gate_checks\"].append({\n            \"check\": \"failure_rate_below_10_percent\",\n            \"passed\": bool(report[\"failure_rate\"] <= 0.10),\n            \"value\": report[\"failure_rate\"],\n        })\n    if not summary.empty:\n        ac = summary[(summary[\"metric\"].astype(str).str.contains(\"effective_rank\", case=False, na=False)) | (summary[\"metric\"].astype(str).str.contains(\"eff_rank\", case=False, na=False))]\n        if not ac.empty:\n            report[\"gate_checks\"].append({\"check\": \"effective_rank_logged\", \"passed\": True, \"rows\": int(len(ac))})\n        val_pred = summary[summary[\"metric\"] == \"val_pred_loss\"]\n        report[\"gate_checks\"].append({\"check\": \"val_prediction_loss_logged\", \"passed\": bool(len(val_pred) > 0), \"rows\": int(len(val_pred))})\n        surf = summary[summary[\"metric\"] == \"normalized_surface\"]\n        report[\"gate_checks\"].append({\"check\": \"routing_surface_logged\", \"passed\": bool(len(surf) > 0), \"rows\": int(len(surf))})\n    return report\n\n\ndef write_markdown_report(report: dict, path: str | Path) -> None:\n    path = Path(path)\n    lines = [\n        \"# GR-CS-HJEPA Phase 1 Pilot Readiness Report\",\n        \"\",\n        report[\"interpretation\"],\n        \"\",\n        f\"Rows analyzed: {report['rows']}\",\n        f\"Studies present: {', '.join(report['studies_present'])}\",\n        f\"Seeds present: {', '.join(map(str, report['seeds_present']))}\",\n        f\"Status counts: {report['status_counts']}\",\n        f\"Failure rate: {report['failure_rate']}\",\n        \"\",\n        \"## Gate checks\",\n    ]\n    for g in report.get(\"gate_checks\", []):\n        lines.append(f\"- {g.get('check')}: {'PASS' if g.get('passed') else 'REVIEW'} ({g})\")\n    lines.extend([\"\", \"## Suggested confirmatory seed counts from pilot variance\", \"\"])\n    if report.get(\"suggested_confirmatory_n\"):\n        for item in report[\"suggested_confirmatory_n\"][:60]:\n            lines.append(f\"- {item}\")\n    else:\n        lines.append(\"- No seed-count suggestions were generated. This usually means too few seeds or missing target metrics.\")\n    lines.extend([\"\", \"## Next actions\", \"\"])\n    for action in report.get(\"next_actions\", []):\n        lines.append(f\"- {action}\")\n    path.write_text(\"\\n\".join(lines))\n\n\ndef analyze_phase1_pilot(results_dir: str | Path, output_dir: str | Path) -> dict:\n    results_dir = Path(results_dir)\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    df = load_pilot_results(results_dir)\n    df.to_csv(output_dir / \"phase1_pilot_loaded_results.csv\", index=False)\n    summary = summarize_numeric(df)\n    summary.to_csv(output_dir / \"phase1_pilot_numeric_summary.csv\", index=False)\n    report = readiness_report(df, summary)\n    save_json(report, output_dir / \"phase1_pilot_readiness_report.json\")\n    write_markdown_report(report, output_dir / \"phase1_pilot_readiness_report.md\")\n    return report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=\"Analyze Phase 1 pilot hardening outputs.\")\n    parser.add_argument(\"--results-dir\", default=\"runs/phase1_pilot\")\n    parser.add_argument(\"--output-dir\", default=\"analysis/phase1_pilot\")\n    args = parser.parse_args()\n    report = analyze_phase1_pilot(args.results_dir, args.output_dir)\n    print(pd.Series({\n        \"rows\": report[\"rows\"],\n        \"studies_present\": \",\".join(report[\"studies_present\"]),\n        \"seeds_present\": \",\".join(map(str, report[\"seeds_present\"])),\n        \"failure_rate\": report[\"failure_rate\"],\n    }).to_string())\n    print(f\"Report written to {Path(args.output_dir) / 'phase1_pilot_readiness_report.md'}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/run_phase1_pilot.py": "from grcshjepa.pilot.runner import main\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/analyze_phase1_pilot.py": "from grcshjepa.pilot.analysis import main\n\nif __name__ == \"__main__\":\n    main()\n", "tests/test_phase1_pilot.py": "\nfrom pathlib import Path\n\nfrom grcshjepa.config import ExperimentConfig\nfrom grcshjepa.pilot.runner import load_yaml\n\n\ndef test_phase1_meta_config_loads():\n    meta = load_yaml(\"configs/phase1_pilot.yaml\")\n    assert \"seed_list\" in meta\n    assert \"study1\" in meta[\"studies\"]\n\n\ndef test_pilot_study_configs_are_experiment_configs():\n    for path in [\"configs/study1_pilot.yaml\", \"configs/study2_pilot.yaml\", \"configs/study3_pilot.yaml\"]:\n        cfg = ExperimentConfig.from_yaml(path)\n        assert cfg.train_samples > 0\n        assert cfg.epochs >= 1\n"}

if WRITE_PHASE1_FILES:
    written = []
    skipped = []
    for rel, content in PHASE1_FILES.items():
        path = Path(rel)
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() and not OVERWRITE_PHASE1_FILES:
            skipped.append(str(path))
            continue
        path.write_text(content)
        written.append(str(path))
    print(f"Wrote {len(written)} Phase 1 files.")
    for p in written:
        print("  wrote", p)
    if skipped:
        print(f"Skipped {len(skipped)} existing files because OVERWRITE_PHASE1_FILES=False")
else:
    print("WRITE_PHASE1_FILES=False. No files written.")

print("\nPhase 1 tree preview:")
for p in [
    "configs/phase1_pilot.yaml",
    "configs/study1_pilot.yaml",
    "configs/study2_pilot.yaml",
    "configs/study3_pilot.yaml",
    "src/grcshjepa/pilot/runner.py",
    "src/grcshjepa/pilot/analysis.py",
    "scripts/run_phase1_pilot.py",
    "scripts/analyze_phase1_pilot.py",
    "tests/test_phase1_pilot.py",
]:
    print(" ", p, "exists=", Path(p).exists())

Wrote 10 Phase 1 files.
  wrote configs/phase1_pilot.yaml
  wrote configs/study1_pilot.yaml
  wrote configs/study2_pilot.yaml
  wrote configs/study3_pilot.yaml
  wrote src/grcshjepa/pilot/__init__.py
  wrote src/grcshjepa/pilot/runner.py
  wrote src/grcshjepa/pilot/analysis.py
  wrote scripts/run_phase1_pilot.py
  wrote scripts/analyze_phase1_pilot.py
  wrote tests/test_phase1_pilot.py

Phase 1 tree preview:
  configs/phase1_pilot.yaml exists= True
  configs/study1_pilot.yaml exists= True
  configs/study2_pilot.yaml exists= True
  configs/study3_pilot.yaml exists= True
  src/grcshjepa/pilot/runner.py exists= True
  src/grcshjepa/pilot/analysis.py exists= True
  scripts/run_phase1_pilot.py exists= True
  scripts/analyze_phase1_pilot.py exists= True
  tests/test_phase1_pilot.py exists= True


In [4]:
# ============================================================
# 4. Install package in editable mode and report environment
# ============================================================
run(f"{sys.executable} -m pip install -q -U pip", cwd=PROJECT_DIR)
run(f"{sys.executable} -m pip install -q -e .[dev]", cwd=PROJECT_DIR)

import torch
import grcshjepa

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception:
    git_commit = "unknown"

env = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_version": torch.version.cuda,
    "grcshjepa_file": grcshjepa.__file__,
    "git_commit": git_commit,
}
print(json.dumps(env, indent=2))
Path("runs").mkdir(exist_ok=True)
Path("runs/phase1_environment.json").write_text(json.dumps(env, indent=2, default=str))

$ /usr/bin/python3 -m pip install -q -U pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.9 MB/s eta 0:00:00

$ /usr/bin/python3 -m pip install -q -e .[dev]

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "gpu_name": "NVIDIA A100-SXM4-80GB",
  "cuda_version": "12.8",
  "grcshjepa_file": null,
  "git_commit": "208e9001dd0364d9bf138a88afe201c936176711"
}


272

In [5]:
# ============================================================
# 5. Run unit tests before pilot experiments
# ============================================================
if RUN_UNIT_TESTS:
    run(f"{sys.executable} -m pytest -q", cwd=PROJECT_DIR)
else:
    print("RUN_UNIT_TESTS=False; skipped pytest.")

$ /usr/bin/python3 -m pytest -q
.................                                                        [100%]
17 passed in 11.75s



In [6]:
# ============================================================
# 6. Run Phase 1 pilot studies
# ============================================================
# This is the first non-smoke phase. It is still pilot work: modest 4-seed runs by default.
# Use MAX_SEEDS=1 above for a quick debugging pass.

if RUN_PILOT_STUDIES:
    cmd = f"{sys.executable} scripts/run_phase1_pilot.py --meta-config configs/phase1_pilot.yaml --output-dir runs/phase1_pilot --studies {STUDIES} --archive"
    if MAX_SEEDS is not None:
        cmd += f" --max-seeds {MAX_SEEDS}"
    run(cmd, cwd=PROJECT_DIR)
else:
    print("RUN_PILOT_STUDIES=False; skipped pilot run.")

$ /usr/bin/python3 scripts/run_phase1_pilot.py --meta-config configs/phase1_pilot.yaml --output-dir runs/phase1_pilot --studies study1,study2,study3 --archive
[phase1] running study1 seed=0 config_hash=a9b26ae68443
[phase1] running study2 seed=0 config_hash=b5bdd7c1ac7c
[phase1] running study3 seed=0 config_hash=0cd2ead00748
[phase1] running study1 seed=1 config_hash=0cf073c34df8
[phase1] running study2 seed=1 config_hash=a932a8dd6c10
[phase1] running study3 seed=1 config_hash=1f80ebbf7f68
[phase1] running study1 seed=2 config_hash=71df39244f60
[phase1] running study2 seed=2 config_hash=104751687efa
[phase1] running study3 seed=2 config_hash=1ce827b9ef98
[phase1] running study1 seed=3 config_hash=31cf70955797
[phase1] running study2 seed=3 config_hash=49e2564b7afd
[phase1] running study3 seed=3 config_hash=f597a6e151c8
[phase1] archived to runs/phase1_pilot.tar.gz
       phase  study         task predictor  seed   status  epochs  train_samples  val_samples  train_loss  train_pred_loss 

In [7]:
# ============================================================
# 7. Analyze Phase 1 pilot outputs and generate readiness package
# ============================================================
if RUN_ANALYSIS:
    run(f"{sys.executable} scripts/analyze_phase1_pilot.py --results-dir runs/phase1_pilot --output-dir analysis/phase1_pilot", cwd=PROJECT_DIR)
else:
    print("RUN_ANALYSIS=False; skipped analysis.")

$ /usr/bin/python3 scripts/analyze_phase1_pilot.py --results-dir runs/phase1_pilot --output-dir analysis/phase1_pilot
rows                                184
studies_present    study1,study2,study3
seeds_present                   0,1,2,3
failure_rate                        0.0
Report written to analysis/phase1_pilot/phase1_pilot_readiness_report.md



In [8]:
# ============================================================
# 8. Display pilot tables in the notebook
# ============================================================
import pandas as pd
from IPython.display import display, Markdown

combined_path = PROJECT_DIR / "runs/phase1_pilot/phase1_pilot_combined_results.csv"
summary_path = PROJECT_DIR / "analysis/phase1_pilot/phase1_pilot_numeric_summary.csv"
report_md_path = PROJECT_DIR / "analysis/phase1_pilot/phase1_pilot_readiness_report.md"

if combined_path.exists():
    combined = pd.read_csv(combined_path)
    print("Combined pilot rows:", len(combined))
    display(combined.head(30))
else:
    print("Combined results not found:", combined_path)

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print("Numeric summary rows:", len(summary))
    # Show the most important diagnostics first.
    important = summary[summary["metric"].isin(["val_pred_loss", "test_acc", "test_mse", "exactish_rate", "normalized_surface", "traffic_degradation", "surface_degradation"])]
    display(important.head(80) if len(important) else summary.head(80))
else:
    print("Summary not found:", summary_path)

if report_md_path.exists():
    display(Markdown(report_md_path.read_text()))
else:
    print("Readiness report not found:", report_md_path)

Combined pilot rows: 184


,phase,study,task,predictor,seed,status,epochs,train_samples,val_samples,train_loss,...,total_length,tube_area,junction_area,surface,delivered_traffic,normalized_surface,gate_entropy,load_gini,surface_degradation,traffic_degradation
0,phase1_pilot,study1,maze,mlp,0,complete,3.0,384.0,128.0,0.480220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,phase1_pilot,study1,maze,spiking,0,complete,3.0,384.0,128.0,0.480090,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,phase1_pilot,study1,sorting,mlp,0,complete,3.0,384.0,128.0,0.480171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,phase1_pilot,study1,sorting,spiking,0,complete,3.0,384.0,128.0,0.478786,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,phase1_pilot,study2,maze_action,NaN,0,complete,NaN,NaN,NaN,1.377946,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,phase1_pilot,study2,sorting_head,NaN,0,complete,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,phase1_pilot,study3,NaN,NaN,0,complete,NaN,NaN,NaN,NaN,...,54.803298,12.065706,2.061302,14.127008,62.851873,0.224767,0.541567,0.612345,NaN,NaN
7,phase1_pilot,study3,NaN,NaN,0,complete,NaN,NaN,NaN,NaN,...,52.907602,11.717642,2.035685,13.753327,61.772514,0.222645,0.518380,0.628254,-0.002122,1.079358
8,phase1_pilot,study3,NaN,NaN,0,complete,NaN,NaN,NaN,NaN,...,49.979161,11.101028,1.988694,13.089722,59.398054,0.220373,0.488680,0.649394,-0.004394,3.453819
9,phase1_pilot,study3,NaN,NaN,0,complete,NaN,NaN,NaN,NaN,...,47.236044,10.338976,1.858852,12.197828,53.157425,0.229466,0.462092,0.661510,0.004699,9.694448


Numeric summary rows: 590


,study,task,predictor,variant,damage_type,damage_level,head_type,metric,n_rows,n_seeds,mean,sd,median,q25,q75,boot_ci_low,boot_ci_high
11,study1,maze,mlp,NaN,NaN,NaN,NaN,val_pred_loss,4,4,0.000092,0.000029,0.000099,0.000076,0.000115,0.000069,0.000116
28,study1,maze,spiking,NaN,NaN,NaN,NaN,val_pred_loss,4,4,0.000052,0.000016,0.000047,0.000041,0.000058,0.000040,0.000066
45,study1,sorting,mlp,NaN,NaN,NaN,NaN,val_pred_loss,4,4,0.001685,0.000467,0.001629,0.001331,0.001983,0.001301,0.002069
61,study1,sorting,spiking,NaN,NaN,NaN,NaN,val_pred_loss,4,4,0.002065,0.000600,0.002333,0.001979,0.002419,0.001485,0.002420
71,study2,maze_action,NaN,NaN,NaN,NaN,classification,test_acc,4,4,0.248047,0.033979,0.253906,0.232422,0.269531,0.218750,0.273438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
396,study3,NaN,NaN,sparsity,spatial,0.05,NaN,traffic_degradation,4,4,3.624578,1.216486,3.587828,2.607021,4.605385,2.576301,4.672856
405,study3,NaN,NaN,sparsity,spatial,0.10,NaN,normalized_surface,4,4,0.213357,0.008261,0.216035,0.210772,0.218620,0.204628,0.219014
408,study3,NaN,NaN,sparsity,spatial,0.10,NaN,surface_degradation,4,4,-0.001160,0.005159,-0.001979,-0.003382,0.000244,-0.005313,0.002760
409,study3,NaN,NaN,sparsity,spatial,0.10,NaN,traffic_degradation,4,4,5.835937,0.736600,5.763370,5.407405,6.191902,5.282112,6.385779


# GR-CS-HJEPA Phase 1 Pilot Readiness Report

Pilot hardening only. These values estimate runtime, variance, failure modes, and metric sanity; they are not confirmatory dissertation results.

Rows analyzed: 184
Studies present: study1, study2, study3
Seeds present: 0, 1, 2, 3
Status counts: {'complete': 184}
Failure rate: 0.0

## Gate checks
- failure_rate_below_10_percent: PASS ({'check': 'failure_rate_below_10_percent', 'passed': True, 'value': 0.0})
- effective_rank_logged: PASS ({'check': 'effective_rank_logged', 'passed': True, 'rows': 8})
- val_prediction_loss_logged: PASS ({'check': 'val_prediction_loss_logged', 'passed': True, 'rows': 4})
- routing_surface_logged: PASS ({'check': 'routing_surface_logged', 'passed': True, 'rows': 40})

## Suggested confirmatory seed counts from pilot variance

- {'study': 'study1', 'task': 'maze', 'predictor': 'mlp', 'metric': 'val_pred_loss', 'pilot_sd': 2.936326171753604e-05, 'target_half_width': 0.02, 'suggested_n_cap20': 2}
- {'study': 'study1', 'task': 'maze', 'predictor': 'spiking', 'metric': 'val_pred_loss', 'pilot_sd': 1.6193473083394405e-05, 'target_half_width': 0.02, 'suggested_n_cap20': 2}
- {'study': 'study1', 'task': 'sorting', 'predictor': 'mlp', 'metric': 'val_pred_loss', 'pilot_sd': 0.0004674879416862808, 'target_half_width': 0.02, 'suggested_n_cap20': 2}
- {'study': 'study1', 'task': 'sorting', 'predictor': 'spiking', 'metric': 'val_pred_loss', 'pilot_sd': 0.0006001528596633972, 'target_half_width': 0.02, 'suggested_n_cap20': 2}
- {'study': 'study2', 'task': 'maze_action', 'head_type': 'classification', 'metric': 'test_acc', 'pilot_sd': 0.033979136329947625, 'target_half_width': 0.03, 'suggested_n_cap20': 5}
- {'study': 'study2', 'task': 'sorting_head', 'head_type': 'regression', 'metric': 'test_mse', 'pilot_sd': 0.02613741263267676, 'target_half_width': 0.02, 'suggested_n_cap20': 7}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.013084813691885534, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.00810977703140924, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 3.041428809013131, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.019795109488163275, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.013374921916900777, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 3.2951727376960616, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.024282983049775813, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'surface_degradation', 'pilot_sd': 0.018084247595365623, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'traffic_degradation', 'pilot_sd': 3.7381506218805955, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'none', 'damage_level': 0.0, 'metric': 'normalized_surface', 'pilot_sd': 0.013078825038054099, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.01495097733880006, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.001984403041944589, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 0.20572034430834069, 'target_half_width': 1.0, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.010238142936722774, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.0049713244983912025, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 2.595772474814195, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.017796339817096458, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'surface_degradation', 'pilot_sd': 0.007686235961608342, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'traffic_degradation', 'pilot_sd': 2.7772645840190875, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.013600299570632988, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.0005864025857408954, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 0.1052544201627694, 'target_half_width': 1.0, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.01369324805192407, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.002826613669516796, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 1.1763763421890927, 'target_half_width': 1.0, 'suggested_n_cap20': 6}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.014746904362718201, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.15, 'metric': 'surface_degradation', 'pilot_sd': 0.004983302250972911, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'euclidean_length', 'damage_type': 'uniform', 'damage_level': 0.15, 'metric': 'traffic_degradation', 'pilot_sd': 1.7283253727869907, 'target_half_width': 1.0, 'suggested_n_cap20': 12}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.004659810564086949, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.002648535471616007, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 6.1814800773582865, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.005972094025460245, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.0031387130113305134, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 8.23318917503843, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.0056689977463399484, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'surface_degradation', 'pilot_sd': 0.003243902768574214, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'load_targeted', 'damage_level': 0.15, 'metric': 'traffic_degradation', 'pilot_sd': 8.424996839201976, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'none', 'damage_level': 0.0, 'metric': 'normalized_surface', 'pilot_sd': 0.0050137837696784425, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.005158279466340439, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.0002921151276274634, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 0.48045041975659464, 'target_half_width': 1.0, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.005647469638851747, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.0014631575832404033, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 13.52882506312665, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.004309985639751154, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'surface_degradation', 'pilot_sd': 0.0007159622203212718, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'spatial', 'damage_level': 0.15, 'metric': 'traffic_degradation', 'pilot_sd': 15.822990958272387, 'target_half_width': 1.0, 'suggested_n_cap20': 20}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'normalized_surface', 'pilot_sd': 0.005027691073317917, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'surface_degradation', 'pilot_sd': 0.00020782543245177724, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.05, 'metric': 'traffic_degradation', 'pilot_sd': 1.9609699450844338, 'target_half_width': 1.0, 'suggested_n_cap20': 15}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'normalized_surface', 'pilot_sd': 0.004924746679739257, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'surface_degradation', 'pilot_sd': 0.000664075918011245, 'target_half_width': 0.05, 'suggested_n_cap20': 2}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.1, 'metric': 'traffic_degradation', 'pilot_sd': 1.5667438405204812, 'target_half_width': 1.0, 'suggested_n_cap20': 10}
- {'study': 'study3', 'variant': 'full_surface', 'damage_type': 'uniform', 'damage_level': 0.15, 'metric': 'normalized_surface', 'pilot_sd': 0.006342571427628849, 'target_half_width': 0.05, 'suggested_n_cap20': 2}

## Next actions

- Inspect failure manifests and rerun only administrative failures under the same seed.
- Replace remaining toy/smoke stand-ins with production architecture modules where needed.
- Freeze model variants, primary endpoints, seed list, hyperparameter budget, and analysis scripts before confirmatory runs.
- Use pilot variance to select 12 to 20 independent confirmatory seeds per primary variant.

In [9]:
# ============================================================
# 9. Archive Phase 1 artifacts to Google Drive if mounted
# ============================================================
import tarfile
from datetime import datetime, timezone

if RUN_ARCHIVE_TO_DRIVE and DRIVE_ARTIFACT_DIR is not None and DRIVE_ARTIFACT_DIR.exists():
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_name = f"grcshjepa_phase1_pilot_{timestamp}.tar.gz"
    archive_path = DRIVE_ARTIFACT_DIR / archive_name
    include_paths = [
        "configs/phase1_pilot.yaml",
        "configs/study1_pilot.yaml",
        "configs/study2_pilot.yaml",
        "configs/study3_pilot.yaml",
        "src/grcshjepa/pilot",
        "scripts/run_phase1_pilot.py",
        "scripts/analyze_phase1_pilot.py",
        "tests/test_phase1_pilot.py",
        "runs/phase1_pilot",
        "analysis/phase1_pilot",
        "runs/phase1_environment.json",
    ]
    DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "w:gz") as tar:
        for rel in include_paths:
            p = PROJECT_DIR / rel
            if p.exists():
                tar.add(p, arcname=rel)
    print("Archived Phase 1 pilot package to:", archive_path)
elif DRIVE_ARTIFACT_DIR is None:
    print("Not in Colab or Drive not configured; no Drive archive written.")
else:
    print("Drive artifact directory does not exist; no Drive archive written:", DRIVE_ARTIFACT_DIR)

Archived Phase 1 pilot package to: /content/drive/MyDrive/grcshjepa_artifacts/grcshjepa_phase1_pilot_20260726T231707Z.tar.gz


In [10]:
# ============================================================
# 10. Git status and suggested commit
# ============================================================
run("git status --short", cwd=PROJECT_DIR, check=False)

print("""
Suggested next commit after reviewing outputs:

  git add configs/*pilot*.yaml src/grcshjepa/pilot scripts/run_phase1_pilot.py scripts/analyze_phase1_pilot.py tests/test_phase1_pilot.py
  git commit -m "Add Phase 1 pilot hardening workflow"
  git push

Do not commit large runs/checkpoints to GitHub. Store those in Drive, object storage, or a release artifact.
""")

$ git status --short
 M src/grcshjepa.egg-info/SOURCES.txt
 M src/grcshjepa/__pycache__/__init__.cpython-312.pyc
 M src/grcshjepa/__pycache__/config.cpython-312.pyc
 M src/grcshjepa/__pycache__/diagnostics.cpython-312.pyc
 M src/grcshjepa/__pycache__/losses.cpython-312.pyc
 M src/grcshjepa/__pycache__/utils.cpython-312.pyc
 M src/grcshjepa/data/__pycache__/__init__.cpython-312.pyc
 M src/grcshjepa/data/__pycache__/datasets.cpython-312.pyc
 M src/grcshjepa/experiments/__pycache__/__init__.cpython-312.pyc
 M src/grcshjepa/experiments/__pycache__/study1.cpython-312.pyc
 M src/grcshjepa/experiments/__pycache__/study2.cpython-312.pyc
 M src/grcshjepa/experiments/__pycache__/study3.cpython-312.pyc
 M src/grcshjepa/models/__pycache__/__init__.cpython-312.pyc
 M src/grcshjepa/models/__pycache__/encoders.cpython-312.pyc
 M src/grcshjepa/models/__pycache__/heads.cpython-312.pyc
 M src/grcshjepa/models/__pycache__/hjepa.cpython-312.pyc
 M src/grcshjepa/models/__pycache__/predictors.cpython-312.py

## How to interpret this notebook after it runs

A successful Phase 1 pilot hardening run means:

1. The repository remains installable.
2. Unit tests pass after the new pilot modules are added.
3. Study 1, Study 2, and Study 3 can run across a small seed list without hidden notebook state.
4. Each seed writes a manifest and CSV metrics.
5. The analysis script can aggregate results and produce a readiness report.
6. The pilot exposes whether variance, failure rates, runtime, anti-collapse metrics, downstream-head learning, and routing-damage metrics are usable.

It does **not** mean any dissertation hypothesis is supported. The next step after reviewing this notebook is to fix production-code weaknesses, decide whether the pilot metrics are meaningful, and then freeze a confirmatory protocol.